# 02 — Spectral Indices & Differential Analysis

Derives NDVI, EVI and NBR from the summer median composites (`src/landsat_composites.py`) using `src/indices.py`, then:

1. Renders the differential maps **ΔNDVI** and **ΔNBR** (T3 − T1, i.e. 2022–2025 minus 1985–1991) for Gallura, Basilicata, Alpi Apuane and Trento and saves them as PNG in `results/maps/`.
2. Builds `results/summary_stats.csv` with the columns Region, Total Area (km²), Expanded Forest (km²), Mean ΔNDVI, Mean ΔNBR.

**Method notes**
- Δ = recent − baseline, so positive values mean greener / denser canopy (for NBR: recovery). This is the opposite sign of the classic burn-severity dNBR (pre − post).
- A pixel is *forest* when its summer NDVI ≥ `FOREST_NDVI_THRESHOLD` (default 0.6). *Expanded forest* = non-forest in T1 and forest in T3.
- Sea and permanent water (ESA WorldCover 2021, class 80 / no data) are masked so the coastal bounding boxes do not dilute the regional means. *Total Area* is the full bounding-box area from `config/areas.json`.
- T1 is Landsat 5 TM and T3 is Landsat 8/9 OLI. No cross-sensor harmonization is applied, so small systematic sensor offsets are part of the Δ.

## 1. Setup

The shared GEE project ID is loaded from `config/project_id.json`. On first use, `ee.Authenticate()` opens the browser sign-in.

In [6]:
import json
import os
import sys
from pathlib import Path

import ee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.indices import add_all_indices
from src.landsat_composites import (
    AREA_KEYS,
    CONFIG_DIR,
    get_median_composite,
    load_areas,
    load_time_periods,
)

In [7]:
import importlib
import src.landsat_composites as landsat_composites

importlib.reload(landsat_composites)
PROJECT_ID = landsat_composites.load_project_id()

try:
    ee.Initialize(project=PROJECT_ID)
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print("Google Earth Engine initialized successfully.")

Google Earth Engine initialized successfully.


## 2. Parameters

In [8]:
BASELINE_PERIOD = "T1"
RECENT_PERIOD = "T3"
FOREST_NDVI_THRESHOLD = 0.6
MASK_WATER = True

STATS_SCALE = 30
MAP_LONG_SIDE_PX = 1600
NODATA = -9999

DELTA_BANDS = {
    "dNDVI": {"label": "ΔNDVI", "limit": 0.4},
    "dNBR": {"label": "ΔNBR", "limit": 0.5},
}

RESULTS_DIR = PROJECT_ROOT / "results"
MAPS_DIR = RESULTS_DIR / "maps"
MAPS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Regions and time periods

In [9]:
areas = load_areas()
config = load_time_periods()
area_info = json.loads((CONFIG_DIR / "areas.json").read_text(encoding="utf-8"))

baseline = config["periods"][BASELINE_PERIOD]
recent = config["periods"][RECENT_PERIOD]
delta_suffix = f"{RECENT_PERIOD}_minus_{BASELINE_PERIOD}"
delta_years = (
    f"{recent['start_year']}–{recent['end_year']} minus "
    f"{baseline['start_year']}–{baseline['end_year']}"
)

print(f"Δ = {RECENT_PERIOD} − {BASELINE_PERIOD} ({delta_years})")
for key in AREA_KEYS:
    print(f"  {key:12s} {area_info[key]['name']:20s} bbox={area_info[key]['bbox']}")

Δ = T3 − T1 (2022–2025 minus 1985–1991)
  gallura      Gallura              bbox=[8.82, 40.43, 9.83, 41.32]
  basilicata   Basilicata           bbox=[15.34, 39.89, 16.87, 41.07]
  alpi_apuane  Alpi Apuane          bbox=[9.85, 43.9, 10.55, 44.35]
  trento       Trento / Trentino    bbox=[10.45, 45.67, 12.25, 46.53]


## 4. Index composites and Δ images

`add_all_indices` needs the sensor band mapping: Landsat 8/9 periods use `L8_L9`, Landsat 5/7 periods use `L5_L7`. Each Δ image holds `dNDVI`, `dNBR` and a binary `expanded_forest` band.

In [10]:
OLI_SENSORS = {"LC08", "LC09"}


def sensor_type_for(period):
    sensors = {collection_id.split("/")[1] for collection_id in period["collection_ids"]}
    if sensors <= OLI_SENSORS:
        return "L8_L9"
    if sensors.isdisjoint(OLI_SENSORS):
        return "L5_L7"
    raise ValueError(f"Period {period['name']!r} mixes OLI and TM/ETM+ collections.")


def index_composite(aoi, period):
    composite = get_median_composite(aoi, period, config)
    return add_all_indices(composite, sensor_type_for(period)).select(["NDVI", "EVI", "NBR"])


def land_mask():
    worldcover = ee.ImageCollection("ESA/WorldCover/v200").first()
    return worldcover.neq(80)


def delta_image(aoi):
    before = index_composite(aoi, baseline)
    after = index_composite(aoi, recent)

    delta = (
        after.select(["NDVI", "NBR"])
        .subtract(before.select(["NDVI", "NBR"]))
        .rename(list(DELTA_BANDS))
        .toFloat()
    )
    forest_before = before.select("NDVI").gte(FOREST_NDVI_THRESHOLD)
    forest_after = after.select("NDVI").gte(FOREST_NDVI_THRESHOLD)
    expanded = forest_after.And(forest_before.Not()).rename("expanded_forest")

    image = delta.addBands(expanded)
    return image.updateMask(land_mask()) if MASK_WATER else image


deltas = {key: delta_image(areas[key]) for key in AREA_KEYS}

## 5. Differential maps (PNG)

Pixels are fetched with `ee.data.computePixels` on a lat/lon grid (longest side = `MAP_LONG_SIDE_PX`), plotted with matplotlib and saved to `results/maps/`. Brown = decrease, green = increase, grey = masked (water or no clear observation).

In [11]:
def fetch_bands(image, bbox, bands):
    xmin, ymin, xmax, ymax = bbox
    pixel_deg = max(xmax - xmin, ymax - ymin) / MAP_LONG_SIDE_PX
    request = {
        "expression": image.select(bands).unmask(NODATA),
        "fileFormat": "NUMPY_NDARRAY",
        "grid": {
            "dimensions": {
                "width": int(round((xmax - xmin) / pixel_deg)),
                "height": int(round((ymax - ymin) / pixel_deg)),
            },
            "affineTransform": {
                "scaleX": pixel_deg,
                "shearX": 0,
                "translateX": xmin,
                "shearY": 0,
                "scaleY": -pixel_deg,
                "translateY": ymax,
            },
            "crsCode": "EPSG:4326",
        },
    }
    pixels = ee.data.computePixels(request)
    return {band: np.where(pixels[band] == NODATA, np.nan, pixels[band]) for band in bands}


def save_delta_map(key, band, array):
    xmin, ymin, xmax, ymax = area_info[key]["bbox"]
    spec = DELTA_BANDS[band]
    cmap = plt.get_cmap("BrBG").copy()
    cmap.set_bad("#d9d9d9")

    fig, ax = plt.subplots(figsize=(9, 7))
    im = ax.imshow(
        np.ma.masked_invalid(array),
        cmap=cmap,
        vmin=-spec["limit"],
        vmax=spec["limit"],
        extent=(xmin, xmax, ymin, ymax),
        interpolation="nearest",
    )
    ax.set_aspect(1 / np.cos(np.deg2rad((ymin + ymax) / 2)))
    ax.set_title(f"{spec['label']} — {area_info[key]['name']}\n{delta_years}")
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")
    fig.colorbar(
        im,
        ax=ax,
        shrink=0.8,
        extend="both",
        label=f"{spec['label']} ({RECENT_PERIOD} − {BASELINE_PERIOD})",
    )

    path = MAPS_DIR / f"{key}_{band}_{delta_suffix}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    return path


for key in AREA_KEYS:
    arrays = fetch_bands(deltas[key], area_info[key]["bbox"], list(DELTA_BANDS))
    for band, array in arrays.items():
        print("Saved", save_delta_map(key, band, array).relative_to(PROJECT_ROOT))

EEException: User memory limit exceeded.

## 6. Summary statistics

Means and expanded-forest area are computed at `STATS_SCALE` m over the (land-masked) pixels of each region. If a region times out, raise `STATS_SCALE` (e.g. 60) or move the reduction to `ee.batch.Export.table`.

In [ ]:
def region_stats(key):
    aoi = areas[key]
    image = deltas[key]
    reduce_kwargs = {"geometry": aoi, "scale": STATS_SCALE, "maxPixels": 1e13, "tileScale": 4}

    means = image.select(list(DELTA_BANDS)).reduceRegion(reducer=ee.Reducer.mean(), **reduce_kwargs)
    expanded_m2 = (
        image.select("expanded_forest")
        .multiply(ee.Image.pixelArea())
        .reduceRegion(reducer=ee.Reducer.sum(), **reduce_kwargs)
    )
    stats = ee.Dictionary({
        "total_m2": aoi.area(maxError=1),
        "expanded_m2": expanded_m2.get("expanded_forest"),
        "dNDVI": means.get("dNDVI"),
        "dNBR": means.get("dNBR"),
    }).getInfo()

    return {
        "Region": area_info[key]["name"],
        "Total Area (km²)": stats["total_m2"] / 1e6,
        "Expanded Forest (km²)": stats["expanded_m2"] / 1e6,
        "Mean ΔNDVI": stats["dNDVI"],
        "Mean ΔNBR": stats["dNBR"],
    }


rows = []
for key in AREA_KEYS:
    print(f"Computing statistics for {area_info[key]['name']}...")
    rows.append(region_stats(key))

summary = pd.DataFrame(rows).round({
    "Total Area (km²)": 2,
    "Expanded Forest (km²)": 2,
    "Mean ΔNDVI": 4,
    "Mean ΔNBR": 4,
})

summary_path = RESULTS_DIR / "summary_stats.csv"
summary.to_csv(summary_path, index=False)
print("Saved", summary_path.relative_to(PROJECT_ROOT))
summary